In [12]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import io
import re
import time
from bs4 import BeautifulSoup
import pandas as pd
import requests
from tqdm.notebook import tqdm

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:153.0) Gecko/20100101 Firefox/153.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

# Mapeamento oficial dos splits de Chicago para o nosso Schema
SPLIT_MAP_CHICAGO = {
    "05K": "Time5K",
    "5K": "Time5K",
    "10K": "Time10K",
    "15K": "Time15K",
    "20K": "Time20K",
    "HALF": "TimeHalf",
    "25K": "Time25K",
    "30K": "Time30K",
    "35K": "Time35K",
    "40K": "Time40K",
    "Finish": "ChipFinish",
    # Splits granulares de milhas (preservados no raw)
    "Mile 20": "TimeMile20",
    "Mile 21": "TimeMile21",
    "Mile 22": "TimeMile22",
    "Mile 23": "TimeMile23",
    "Mile 24": "TimeMile24",
    "Mile 25": "TimeMile25",
    "Mile 25.2": "TimeMile25_2",
}


def parse_splits_table(html_content: str) -> dict:
    """Extrai os splits de tempo a partir da página HTML de detalhes do atleta."""
    splits = {}
    try:
        tables = pd.read_html(io.StringIO(html_content), flavor="lxml")
        # Procura a tabela que contém a coluna 'Split'
        for tbl in tables:
            if "Split" in tbl.columns and "Time" in tbl.columns:
                for _, row in tbl.iterrows():
                    split_name = str(row["Split"]).strip()
                    split_time = str(row["Time"]).strip()
                    if split_name in SPLIT_MAP_CHICAGO:
                        col_canonical = SPLIT_MAP_CHICAGO[split_name]
                        splits[col_canonical] = split_time
                break
    except Exception:
        pass
    return splits

In [13]:
ANO_PILOTO = 2024
BASE_URL_CHICAGO = f"https://chicago-history.r.mikatiming.com/{ANO_PILOTO}/"

params_list = {
    "pid": "list",
    "page": "1",
    "event": "MAR_9TGG963812D",
    "search[sex]": "%",
    "search[age_class]": "%",
}

print(f"⏳ Baixando lista inicial de Chicago {ANO_PILOTO}...")
res_list = requests.get(BASE_URL_CHICAGO, headers=HEADERS, params=params_list)
soup_list = BeautifulSoup(res_list.text, "html.parser")

athletes_meta = []

for item in soup_list.select("li.list-group-item.row"):
    if "list-group-header" in item.get("class", []):
        continue

    # Classificações
    pl_overall = item.select_one("div.list-field.type-place.place-secondary")
    pl_gender = item.select_one("div.list-field.type-place.place-primary")

    # Nome, País e Link
    name_tag = item.select_one("h4.list-field.type-fullname a")
    if not name_tag:
        continue

    full_text = name_tag.get_text(strip=True)
    href = name_tag["href"]

    nation_m = re.search(r"\(([A-Z]{3})\)$", full_text)
    nation = nation_m.group(1) if nation_m else None
    name_clean = (
        re.sub(r"\s*\([A-Z]{3}\)$", "", full_text).strip()
        if full_text
        else None
    )

    # Função auxiliar de limpeza
    def get_val(selector):
        el = item.select_one(selector)
        if not el:
            return None
        for lab in el.select(".list-label"):
            lab.decompose()
        return el.get_text(strip=True)

    bib = get_val("div.list-field.type-field")
    division = get_val("div.list-field.type-age_class")
    finish_netto = get_val("div.list-field.type-time")

    # URL completa de detalhes
    detail_full_url = (
        BASE_URL_CHICAGO + href
        if not href.startswith("http")
        else href
    )

    athletes_meta.append(
        {
            "EventYear": ANO_PILOTO,
            "RankOverAll": (
                pl_overall.get_text(strip=True) if pl_overall else None
            ),
            "RankOverGender": (
                pl_gender.get_text(strip=True) if pl_gender else None
            ),
            "FormattedFullName": name_clean,
            "CountryOfCTZName": nation,
            "FullBibNumber": bib,
            "AwardsDivisionShortDesc": division,
            "ChipFinish_List": finish_netto,
            "DetailURL": detail_full_url,
        }
    )

df_pilot = pd.DataFrame(athletes_meta)
print(
    f"✅ Metadados extraídos: {len(df_pilot)} atletas prontos para enriquecimento de splits."
)
display(df_pilot.head(3))

⏳ Baixando lista inicial de Chicago 2024...
✅ Metadados extraídos: 25 atletas prontos para enriquecimento de splits.


,EventYear,RankOverAll,RankOverGender,FormattedFullName,CountryOfCTZName,FullBibNumber,AwardsDivisionShortDesc,ChipFinish_List,DetailURL
0,2024,428,1,"Caswell, Jake",USA,459,25-29,02:38:05,https://chicago-history.r.mikatiming.com/2024/...
1,2024,1,1,"Kiptum, Kelvin",KEN,2,20-24,02:00:35,https://chicago-history.r.mikatiming.com/2024/...
2,2024,737,1,"Tellez, Oscar",USA,22195,–,02:43:22,https://chicago-history.r.mikatiming.com/2024/...


In [14]:
def fetch_athlete_splits(athlete_record: dict, session: requests.Session) -> dict:
    """Faz a requisição individual e enriquece o registro com os splits de 5k e milhas."""
    url = athlete_record["DetailURL"]
    merged_record = athlete_record.copy()
    try:
        r = session.get(url, headers=HEADERS, timeout=10)
        if r.status_code == 200:
            splits_data = parse_splits_table(r.text)
            merged_record.update(splits_data)
    except Exception as e:
        pass
    return merged_record


print(f"🚀 Iniciando teste Multithreading ({len(df_pilot)} atletas)...")
t_inicio = time.time()

# Sessão com pool de conexões otimizado
session = requests.Session()
adapter = requests.adapters.HTTPAdapter(pool_connections=20, pool_maxsize=20)
session.mount("https://", adapter)

enriched_records = []
max_threads = 15  # Número de threads concorrentes

with ThreadPoolExecutor(max_workers=max_threads) as executor:
    # Dispara as requisições em paralelo
    futures = [
        executor.submit(fetch_athlete_splits, row, session)
        for row in athletes_meta
    ]

    for f in tqdm(as_completed(futures), total=len(futures), desc="Splits Concorrentes"):
        enriched_records.append(f.result())

t_total = time.time() - t_inicio
velocidade = len(enriched_records) / t_total

print(f"\n⏱️ Tempo Total: {t_total:.2f} segundos")
print(f"⚡ Velocidade Atingida: {velocidade:.1f} atletas/segundo")

df_chicago_pilot_full = pd.DataFrame(enriched_records)

🚀 Iniciando teste Multithreading (25 atletas)...


Splits Concorrentes:   0%|          | 0/25 [00:00<?, ?it/s]


⏱️ Tempo Total: 1.77 segundos
⚡ Velocidade Atingida: 14.1 atletas/segundo


In [15]:
# Remove a coluna de URL temporária para visualização
colunas_exibicao = [
    "RankOverAll",
    "FormattedFullName",
    "CountryOfCTZName",
    "Time5K",
    "Time10K",
    "TimeHalf",
    "Time30K",
    "TimeMile20",
    "Time35K",
    "Time40K",
    "TimeMile25",
    "ChipFinish",
]

colunas_validas = [
    c for c in colunas_exibicao if c in df_chicago_pilot_full.columns
]
print("--- DATAFRAME DE CHICAGO COM SPLITS E MILHAS EXTRAÍDOS ---")
display(df_chicago_pilot_full[colunas_validas].head(5))

--- DATAFRAME DE CHICAGO COM SPLITS E MILHAS EXTRAÍDOS ---


,RankOverAll,FormattedFullName,CountryOfCTZName,Time5K,Time10K,TimeHalf,Time30K,Time35K,Time40K,ChipFinish
0,2,"Kipruto, Benson",KEN,00:14:26,00:29:12,01:02:21,01:28:09,01:42:54,01:57:35,02:04:02
1,1,"Kiptum, Kelvin",KEN,00:14:26,00:28:42,01:00:48,01:26:31,01:40:22,01:54:23,02:00:35
2,428,"Caswell, Jake",USA,00:18:11,00:36:35,01:17:45,01:51:04,02:10:07,02:29:40,02:38:05
3,499,"Grant, Jonah",USA,00:18:28,00:38:30,01:19:54,01:53:35,02:11:52,02:30:35,02:39:12
4,24,"Chepngetich, Ruth",KEN,00:15:42,00:31:05,01:05:42,01:34:10,01:50:48,02:07:52,02:15:37


In [16]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import io
import re
import pandas as pd
import requests
from tqdm.notebook import tqdm

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:153.0) Gecko/20100101 Firefox/153.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}


def parse_athlete_full_raw(html_content: str, base_record: dict) -> dict:
    """Extrai 100% dos dados disponíveis em todas as tabelas da página de detalhes do atleta."""
    raw_data = base_record.copy()

    try:
        tables = pd.read_html(io.StringIO(html_content), flavor="lxml")

        for tbl in tables:
            # 1. Captura da Tabela de Metadados / Cidade / Start Time
            if len(tbl.columns) == 2 and tbl.columns[0] == 0:
                for _, row in tbl.iterrows():
                    key = str(row[0]).strip().lower()
                    val = str(row[1]).strip()

                    if "city" in key or "state" in key:
                        raw_data["City_State"] = val
                    elif "start time" in key:
                        raw_data["Start_Time"] = val
                    elif "place age group" in key or "division" in key:
                        raw_data["Place_Age_Group"] = val

            # 2. Captura Dinâmica da Tabela de Splits (Km + Milhas + Deltas)
            elif "Split" in tbl.columns and "Time" in tbl.columns:
                for _, row in tbl.iterrows():
                    split_raw = str(row["Split"]).strip()
                    # Normaliza o nome do split para virar uma coluna válida
                    split_key = (
                        split_raw.replace(" ", "_")
                        .replace(".", "_")
                        .replace("/", "_")
                    )

                    # Tempo líquido acumulado no tapete
                    raw_data[f"Time_{split_key}"] = (
                        str(row["Time"]).strip()
                        if pd.notnull(row.get("Time"))
                        else None
                    )

                    # Delta/tempo gasto no segmento
                    if "Diff" in tbl.columns and pd.notnull(row.get("Diff")):
                        raw_data[f"Diff_{split_key}"] = str(
                            row["Diff"]
                        ).strip()

                    # Hora do dia da passagem
                    if "Time Of Day" in tbl.columns and pd.notnull(
                        row.get("Time Of Day")
                    ):
                        raw_data[f"TimeOfDay_{split_key}"] = str(
                            row["Time Of Day"]
                        ).strip()

    except Exception:
        pass

    return raw_data

In [17]:
def fetch_full_raw_worker(athlete: dict, session: requests.Session) -> dict:
    url = athlete["DetailURL"]
    try:
        r = session.get(url, headers=HEADERS, timeout=12)
        if r.status_code == 200:
            return parse_athlete_full_raw(r.text, athlete)
    except Exception:
        pass
    return athlete


print("🚀 Extraindo dados brutos completos (splits + deltas + horários)...")

session = requests.Session()
adapter = requests.adapters.HTTPAdapter(pool_connections=15, pool_maxsize=15)
session.mount("https://", adapter)

raw_enriched_list = []

with ThreadPoolExecutor(max_workers=15) as executor:
    futures = [
        executor.submit(fetch_full_raw_worker, ath, session)
        for ath in athletes_meta
    ]

    for f in tqdm(as_completed(futures), total=len(futures), desc="Raw Deep Scraping"):
        raw_enriched_list.append(f.result())

df_chicago_raw_test = pd.DataFrame(raw_enriched_list)

print(
    f"\n✅ Concluído! O DataFrame Raw gerou {df_chicago_raw_test.shape[1]} colunas distintas."
)
print("\n--- TODAS AS COLUNAS CAPTURADAS NO RAW LAKE ---")
for i, col in enumerate(df_chicago_raw_test.columns):
    print(f"{i+1:02d}. {col}")

🚀 Extraindo dados brutos completos (splits + deltas + horários)...


Raw Deep Scraping:   0%|          | 0/25 [00:00<?, ?it/s]


✅ Concluído! O DataFrame Raw gerou 42 colunas distintas.

--- TODAS AS COLUNAS CAPTURADAS NO RAW LAKE ---
01. EventYear
02. RankOverAll
03. RankOverGender
04. FormattedFullName
05. CountryOfCTZName
06. FullBibNumber
07. AwardsDivisionShortDesc
08. ChipFinish_List
09. DetailURL
10. City_State
11. Place_Age_Group
12. Start_Time
13. Time_05K
14. Diff_05K
15. TimeOfDay_05K
16. Time_10K
17. Diff_10K
18. TimeOfDay_10K
19. Time_15K
20. Diff_15K
21. TimeOfDay_15K
22. Time_20K
23. Diff_20K
24. TimeOfDay_20K
25. Time_HALF
26. Diff_HALF
27. TimeOfDay_HALF
28. Time_25K
29. Diff_25K
30. TimeOfDay_25K
31. Time_30K
32. Diff_30K
33. TimeOfDay_30K
34. Time_35K
35. Diff_35K
36. TimeOfDay_35K
37. Time_40K
38. Diff_40K
39. TimeOfDay_40K
40. Time_Finish
41. Diff_Finish
42. TimeOfDay_Finish


In [18]:
# Prévia dos primeiros 3 corredores com todas as colunas
df_chicago_raw_test.head(3)

,EventYear,RankOverAll,RankOverGender,FormattedFullName,CountryOfCTZName,FullBibNumber,AwardsDivisionShortDesc,ChipFinish_List,DetailURL,City_State,Place_Age_Group,Start_Time,Time_05K,Diff_05K,TimeOfDay_05K,Time_10K,Diff_10K,TimeOfDay_10K,Time_15K,Diff_15K,TimeOfDay_15K,Time_20K,Diff_20K,TimeOfDay_20K,Time_HALF,Diff_HALF,TimeOfDay_HALF,Time_25K,Diff_25K,TimeOfDay_25K,Time_30K,Diff_30K,TimeOfDay_30K,Time_35K,Diff_35K,TimeOfDay_35K,Time_40K,Diff_40K,TimeOfDay_40K,Time_Finish,Diff_Finish,TimeOfDay_Finish
0,2024,1009,4,"Bugarin, Steven",USA,1086,35-39,02:46:47,https://chicago-history.r.mikatiming.com/2024/...,Chicago,1,07:32:15,00:19:35,19:35,07:51:50AM,00:39:20,19:45,08:11:34AM,00:59:07,19:47,08:31:22AM,01:18:39,19:32,08:50:54AM,01:22:56,04:17,08:55:11AM,01:38:00,15:04,09:10:15AM,01:57:25,19:25,09:29:40AM,02:17:06,19:41,09:49:20AM,02:37:42,20:36,10:09:57AM,02:46:47,09:05,10:19:02AM
1,2024,4,4,"Korir, John",KEN,7,25-29,02:05:09,https://chicago-history.r.mikatiming.com/2024/...,Eldoret,1,07:30:02,00:14:27,14:27,07:44:28AM,00:29:13,14:46,07:59:15AM,00:44:06,14:53,08:14:08AM,00:59:05,14:59,08:29:06AM,01:02:21,03:16,08:32:23AM,01:13:52,11:31,08:43:54AM,01:28:08,14:16,08:58:10AM,01:42:54,14:46,09:12:56AM,01:58:05,15:11,09:28:07AM,02:05:09,07:04,09:35:11AM
2,2024,21,1,"Hassan, Sifan",NED,108,30-34,02:13:44,https://chicago-history.r.mikatiming.com/2024/...,Nijmegen,1,07:30:02,00:15:43,15:43,07:45:45AM,00:31:05,15:22,08:01:07AM,00:46:37,15:32,08:16:39AM,01:02:16,15:39,08:32:18AM,01:05:48,03:32,08:35:50AM,01:18:06,12:18,08:48:07AM,01:34:00,15:54,09:04:02AM,01:50:17,16:17,09:20:18AM,02:06:36,16:19,09:36:38AM,02:13:44,07:08,09:43:46AM


In [19]:
import io
import re
import pandas as pd
import requests
from bs4 import BeautifulSoup

# URL oficial da edição mais recente
ANO_RECENTE = 2025
URL_LIVE = f"https://results.chicagomarathon.com/{ANO_RECENTE}/"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:153.0) Gecko/20100101 Firefox/153.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

print(f"🔍 Testando conexão com o portal recente: {URL_LIVE}")

# 1. Detecta o EventCode na página inicial do portal recente
res_init = requests.get(URL_LIVE, headers=HEADERS, params={"pid": "list"})
print(f"Status HTTP: {res_init.status_code}")

soup_init = BeautifulSoup(res_init.text, "html.parser")
event_select = soup_init.find("select", {"name": "event"})
event_code_2025 = None

if event_select:
    print("\n--- Eventos encontrados no seletor de 2025 ---")
    for opt in event_select.find_all("option"):
        val = opt.get("value", "")
        txt = opt.text.strip()
        print(f"• Código: '{val}' | Nome: '{txt}'")
        if "marathon" in txt.lower() or val.startswith("MAR") or "2025" in val:
            if not event_code_2025:
                event_code_2025 = val

print(f"\n🎯 EventCode selecionado para 2025: '{event_code_2025}'")

# 2. Pede a página 1 da listagem (Masculino)
params_2025 = {
    "pid": "list",
    "page": "1",
    "search[sex]": "M",
    "search[age_class]": "%",
}
if event_code_2025:
    params_2025["event"] = event_code_2025

res_list_2025 = requests.get(URL_LIVE, headers=HEADERS, params=params_2025)
soup_list_2025 = BeautifulSoup(res_list_2025.text, "html.parser")

cards_2025 = soup_list_2025.select("li.list-group-item.row")
print(
    f"📊 Total de cards de atletas encontrados na página 1: {len(cards_2025)}"
)

# 3. Testa o parsing do primeiro atleta da lista
primeiro_card = None
for c in cards_2025:
    if "list-group-header" not in c.get("class", []):
        primeiro_card = c
        break

if primeiro_card:
    name_tag = primeiro_card.select_one("h4.list-field.type-fullname a")
    finish_tag = primeiro_card.select_one("div.list-field.type-time")
    bib_tag = primeiro_card.select_one("div.list-field.type-field")

    # Limpeza de labels
    if finish_tag and finish_tag.select_one(".list-label"):
        finish_tag.select_one(".list-label").decompose()
    if bib_tag and bib_tag.select_one(".list-label"):
        bib_tag.select_one(".list-label").decompose()

    print("\n--- PRIMEIRO ATLETA ENCONTRADO EM 2025 ---")
    print(f"Nome: {name_tag.get_text(strip=True) if name_tag else '-'}")
    print(f"BIB: {bib_tag.get_text(strip=True) if bib_tag else '-'}")
    print(f"Finish: {finish_tag.get_text(strip=True) if finish_tag else '-'}")
    print(f"Link Detalhes: {name_tag['href'] if name_tag else '-'}")

    # 4. Testa a captura profunda de splits na página de detalhes de 2025
    if name_tag and name_tag.get("href"):
        href = name_tag["href"]
        url_detalhe_2025 = (
            URL_LIVE + href if not href.startswith("http") else href
        )
        print(
            f"\n⏳ Buscando splits na página de detalhes de 2025:\n{url_detalhe_2025}"
        )

        res_det = requests.get(url_detalhe_2025, headers=HEADERS)
        tabelas_2025 = pd.read_html(io.StringIO(res_det.text), flavor="lxml")

        for tbl in tabelas_2025:
            if "Split" in tbl.columns and "Time" in tbl.columns:
                print("\n✅ TABELA DE SPLITS DE 2025 CAPTURADA COM SUCESSO:")
                display(tbl.head(10))
                break

🔍 Testando conexão com o portal recente: https://results.chicagomarathon.com/2025/
Status HTTP: 200

🎯 EventCode selecionado para 2025: 'None'
📊 Total de cards de atletas encontrados na página 1: 26

--- PRIMEIRO ATLETA ENCONTRADO EM 2025 ---
Nome: Kiplimo, Jacob (UGA)
BIB: 6
Finish: 01:00:16
Link Detalhes: ?content=detail&fpid=list&pid=list&idp=9TGG96382C8702&lang=EN_CAP&event=MAR&search%5Bsex%5D=M&search%5Bage_class%5D=%25&search_event=

⏳ Buscando splits na página de detalhes de 2025:
https://results.chicagomarathon.com/2025/?content=detail&fpid=list&pid=list&idp=9TGG96382C8702&lang=EN_CAP&event=MAR&search%5Bsex%5D=M&search%5Bage_class%5D=%25&search_event=

✅ TABELA DE SPLITS DE 2025 CAPTURADA COM SUCESSO:


,Split,Time Of Day,Time,Diff,min/km,km/h,min/mile,miles/h
0,05K,07:43:59AM,00:13:58,13:58,02:48,21.49,04:30,13.36
1,10K,07:58:26AM,00:28:25,14:27,02:54,20.75,04:40,12.90
2,15K,08:12:41AM,00:42:41,14:16,02:52,21.05,04:36,13.08
3,20K,08:27:06AM,00:57:06,14:25,02:53,20.81,04:39,12.93
4,HALF,08:30:17AM,01:00:16,03:10,02:54,20.71,04:40,12.87
5,25K,08:41:13AM,01:11:12,10:56,02:49,21.42,04:31,13.31
6,30K,08:55:32AM,01:25:31,14:19,02:52,20.96,04:37,13.03
7,Mile 20,09:01:42AM,01:31:41,06:10,02:50,21.19,04:34,13.17
8,Mile 21,09:06:20AM,01:36:19,04:38,02:53,20.84,04:39,12.95
9,35K,09:09:53AM,01:39:53,03:34,02:57,20.43,04:44,12.69


In [20]:
from bs4 import BeautifulSoup
import pandas as pd
import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:153.0) Gecko/20100101 Firefox/153.0"
}


def descobrir_event_code_ano(ano: int) -> dict:
    url_base = (
        f"https://results.chicagomarathon.com/{ano}/"
        if ano >= 2025
        else f"https://chicago-history.r.mikatiming.com/{ano}/"
    )

    event_code = None
    event_name = None

    # Tenta obter a página inicial sem parâmetros onde o dropdown de eventos reside
    try:
        r = requests.get(url_base, headers=HEADERS, timeout=6)
        soup = BeautifulSoup(r.text, "html.parser")

        # Procura seletores <select name="event"> ou links com event=
        event_select = soup.find("select", {"name": "event"})
        if event_select:
            for opt in event_select.find_all("option"):
                val = opt.get("value", "")
                txt = opt.text.strip().lower()
                if "marathon" in txt or val.startswith("MAR"):
                    event_code = val
                    event_name = opt.text.strip()
                    break
            if not event_code and event_select.find_all("option"):
                opt = event_select.find_all("option")[0]
                event_code = opt.get("value")
                event_name = opt.text.strip()

        # Se não achou no select, procura em links <a> ou inputs hidden
        if not event_code:
            input_event = soup.find("input", {"name": "event"})
            if input_event:
                event_code = input_event.get("value")

    except Exception as e:
        event_name = str(e)[:30]

    return {
        "Ano": ano,
        "EventCode": event_code,
        "NomeEvento": event_name,
        "URL": url_base,
    }


print("🔍 Mapeando EventCodes reais de Chicago (1996 a 2025)...\n")
mapa_eventos = []
for ano in range(1996, 2026):
    info = descobrir_event_code_ano(ano)
    mapa_eventos.append(info)
    print(f"• Ano {ano}: Code='{info['EventCode']}' | Nome='{info['NomeEvento']}'")

df_mapa = pd.DataFrame(mapa_eventos)

🔍 Mapeando EventCodes reais de Chicago (1996 a 2025)...

• Ano 1996: Code='MAR_9TGG963817D' | Nome='Marathon'
• Ano 1997: Code='MAR_9TGG963817D' | Nome='Marathon'
• Ano 1998: Code='MAR_9TGG963817D' | Nome='Marathon'
• Ano 1999: Code='MAR_9TGG963817D' | Nome='Marathon'
• Ano 2000: Code='MAR_9TGG963817D' | Nome='Marathon'
• Ano 2001: Code='MAR_9TGG963817D' | Nome='Marathon'
• Ano 2002: Code='MAR_9TGG963817D' | Nome='Marathon'
• Ano 2003: Code='MAR_9TGG963817D' | Nome='Marathon'
• Ano 2004: Code='MAR_9TGG963817D' | Nome='Marathon'
• Ano 2005: Code='MAR_9TGG963817D' | Nome='Marathon'
• Ano 2006: Code='MAR_9TGG963817D' | Nome='Marathon'
• Ano 2007: Code='MAR_9TGG963817D' | Nome='Marathon'
• Ano 2008: Code='MAR_9TGG963817D' | Nome='Marathon'
• Ano 2009: Code='MAR_9TGG963817D' | Nome='Marathon'
• Ano 2010: Code='MAR_9TGG963817D' | Nome='Marathon'
• Ano 2011: Code='MAR_9TGG963817D' | Nome='Marathon'
• Ano 2012: Code='MAR_9TGG963817D' | Nome='Marathon'
• Ano 2013: Code='MAR_9TGG963817D' | Nome=

In [21]:
import re
from bs4 import BeautifulSoup
import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:153.0) Gecko/20100101 Firefox/153.0"
}


def checar_edicao_chicago(ano: int):
    url_base = (
        f"https://results.chicagomarathon.com/{ano}/"
        if ano >= 2025
        else f"https://chicago-history.r.mikatiming.com/{ano}/"
    )
    event_code = "MAR" if ano >= 2025 else "MAR_9TGG963817D"

    params = {
        "pid": "list",
        "page": "1",
        "event": event_code,
        "search[sex]": "%",
        "search[age_class]": "%",
    }

    r = requests.get(url_base, headers=HEADERS, params=params, timeout=8)
    soup = BeautifulSoup(r.text, "html.parser")

    # Contagem de resultados
    info_text = soup.find(
        class_=lambda c: c
        and any(k in str(c) for k in ["str_num", "list-info"])
    )
    total_str = info_text.get_text(strip=True) if info_text else "Não indicado"

    # Quantidade de cards na página 1
    cards = len(soup.select("li.list-group-item.row"))

    print(
        f"• Ano {ano} ({url_base}): Header='{total_str}' | Cards Página 1: {cards} | EventCode: {event_code}"
    )


print("🔍 Testando contagem em edições chave...\n")
for ano_teste in [1996, 2010, 2019, 2024, 2025]:
    checar_edicao_chicago(ano_teste)

🔍 Testando contagem em edições chave...

• Ano 1996 (https://chicago-history.r.mikatiming.com/1996/): Header='52064 Results|Official Results' | Cards Página 1: 26 | EventCode: MAR_9TGG963817D
• Ano 2010 (https://chicago-history.r.mikatiming.com/2010/): Header='52064 Results|Official Results' | Cards Página 1: 26 | EventCode: MAR_9TGG963817D
• Ano 2019 (https://chicago-history.r.mikatiming.com/2019/): Header='52064 Results|Official Results' | Cards Página 1: 26 | EventCode: MAR_9TGG963817D
• Ano 2024 (https://chicago-history.r.mikatiming.com/2024/): Header='52064 Results|Official Results' | Cards Página 1: 26 | EventCode: MAR_9TGG963817D
• Ano 2025 (https://results.chicagomarathon.com/2025/): Header='Official Results' | Cards Página 1: 26 | EventCode: MAR


In [22]:
from bs4 import BeautifulSoup
import pandas as pd
import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:153.0) Gecko/20100101 Firefox/153.0"
}


def checar_vencedor(ano: int, dominio: str):
    url = f"https://{dominio}/{ano}/"
    params = {"pid": "list", "page": "1", "search[sex]": "M"}
    try:
        r = requests.get(url, headers=HEADERS, params=params, timeout=6)
        if r.status_code != 200:
            return None, f"HTTP {r.status_code}"

        soup = BeautifulSoup(r.text, "html.parser")
        card = soup.select_one("li.list-group-item.row")
        if card and "list-group-header" not in card.get("class", []):
            name_tag = card.select_one("h4.list-field.type-fullname a")
            time_tag = card.select_one("div.list-field.type-time")
            if time_tag and time_tag.select_one(".list-label"):
                time_tag.select_one(".list-label").decompose()

            nome = name_tag.get_text(strip=True) if name_tag else "-"
            tempo = time_tag.get_text(strip=True) if time_tag else "-"
            return nome, tempo
    except Exception as e:
        return None, str(e)[:30]
    return None, "Sem card"


print("🔍 Testando vencedores nos dois domínios...\n")

for ano in [1996, 1999, 2010, 2018, 2023, 2024, 2025]:
    # Testa no chicago-history
    nome_hist, tempo_hist = checar_vencedor(
        ano, "chicago-history.r.mikatiming.com"
    )
    # Testa no results.chicagomarathon
    nome_res, tempo_res = checar_vencedor(ano, "results.chicagomarathon.com")

    print(f"• Ano {ano}:")
    print(f"   [chicago-history]        -> {nome_hist} ({tempo_hist})")
    print(f"   [results.chicagomarathon] -> {nome_res} ({tempo_res})")

🔍 Testando vencedores nos dois domínios...

• Ano 1996:
   [chicago-history]        -> None (Sem card)
   [results.chicagomarathon] -> None (Sem card)
• Ano 1999:
   [chicago-history]        -> None (Sem card)
   [results.chicagomarathon] -> None (Sem card)
• Ano 2010:
   [chicago-history]        -> None (Sem card)
   [results.chicagomarathon] -> None (Sem card)
• Ano 2018:
   [chicago-history]        -> None (Sem card)
   [results.chicagomarathon] -> None (Sem card)
• Ano 2023:
   [chicago-history]        -> None (Sem card)
   [results.chicagomarathon] -> None (Sem card)
• Ano 2024:
   [chicago-history]        -> None (Sem card)
   [results.chicagomarathon] -> None (Sem card)
• Ano 2025:
   [chicago-history]        -> None (Sem card)
   [results.chicagomarathon] -> None (Sem card)


In [23]:
from bs4 import BeautifulSoup
import pandas as pd
import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:153.0) Gecko/20100101 Firefox/153.0"
}

ROOT_URL = "https://chicago-history.r.mikatiming.com/"

print(f"🔍 Consultando a página raiz do histórico: {ROOT_URL}\n")
res = requests.get(ROOT_URL, headers=HEADERS)
soup = BeautifulSoup(res.text, "html.parser")

print(f"Status HTTP: {res.status_code}")
print(f"Título da Página: {soup.title.text if soup.title else 'Sem título'}\n")

# 1. Procura dropdowns de anos ou reuniões (<select>)
selects = soup.find_all("select")
print(f"Dropdowns (<select>) encontrados: {len(selects)}")
for s in selects:
    print(f"• Select Name: '{s.get('name')}' | ID: '{s.get('id')}'")
    for opt in s.find_all("option")[:5]:  # Primeiras 5 opções
        print(f"   -> Value: '{opt.get('value')}' | Texto: '{opt.text.strip()}'")

# 2. Procura links de anos na página (ex: 2023, 2022, 2021...)
print("\n--- Links com anos encontrados na página inicial ---")
year_links = {}
for a in soup.find_all("a", href=True):
    txt = a.text.strip()
    href = a["href"]
    # Se o texto for um ano entre 1996 e 2025
    if txt.isdigit() and 1990 <= int(txt) <= 2026:
        year_links[int(txt)] = href
        print(f"• Ano {txt} -> Link: '{href}'")

# 3. Se não achou links por texto puro, procura em divs ou botões
if not year_links:
    for a in soup.find_all("a", href=True):
        if any(
            y in a["href"] for y in ["mid=", "year=", "event=", "20", "19"]
        ):
            print(f"Link suspeito: '{a.text.strip()}' -> '{a['href']}'")

🔍 Consultando a página raiz do histórico: https://chicago-history.r.mikatiming.com/

Status HTTP: 200
Título da Página: Bank of America Chicago Marathon

Dropdowns (<select>) encontrados: 10
• Select Name: 'event_main_group' | ID: 'simple-search-event_main_group'
   -> Value: 'ALL' | Texto: 'All'
   -> Value: '2024' | Texto: '2024'
   -> Value: '2023' | Texto: '2023'
   -> Value: '2022' | Texto: '2022'
   -> Value: '2021' | Texto: '2021'
• Select Name: 'event' | ID: 'simple-search-event'
   -> Value: 'ALL_EVENT_GROUP_2024' | Texto: 'All 2024'
   -> Value: 'MAR_9TGG963817D' | Texto: 'Marathon'
   -> Value: 'HCL_9TGG963817D' | Texto: 'Exhibition Handcycle'
   -> Value: 'WHL_9TGG963817D' | Texto: 'Wheelchair'
   -> Value: 'DT_9TGG963817D' | Texto: 'Duo Teams'
• Select Name: 'event_main_group' | ID: 'advanced-search-event_main_group'
   -> Value: 'ALL' | Texto: 'All'
   -> Value: '2024' | Texto: '2024'
   -> Value: '2023' | Texto: '2023'
   -> Value: '2022' | Texto: '2022'
   -> Value: '20

In [24]:
from bs4 import BeautifulSoup
import requests

ROOT_URL = "https://chicago-history.r.mikatiming.com/"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:153.0) Gecko/20100101 Firefox/153.0"
}

res = requests.get(ROOT_URL, headers=HEADERS)
soup = BeautifulSoup(res.text, "html.parser")

select_group = soup.find("select", {"name": "event_main_group"})
anos_disponiveis = []

if select_group:
    for opt in select_group.find_all("option"):
        val = opt.get("value", "")
        txt = opt.text.strip()
        if val.isdigit():
            anos_disponiveis.append(int(val))

anos_disponiveis = sorted(anos_disponiveis)
print("📅 ANOS HISTÓRICOS CONFIRMADOS NO CHICAGOMARATHON:")
print(f"Período: {anos_disponiveis[0]} a {anos_disponiveis[-1]}")
print(f"Total de edições mapeadas: {len(anos_disponiveis)}")
print(f"Lista completa: {anos_disponiveis}")

📅 ANOS HISTÓRICOS CONFIRMADOS NO CHICAGOMARATHON:
Período: 1989 a 2024
Total de edições mapeadas: 35
Lista completa: [1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023, 2024]


In [25]:
import io
import pandas as pd


def testar_ano_com_group(ano: int):
    url = f"https://chicago-history.r.mikatiming.com/2024/"
    params = {
        "pid": "list",
        "event_main_group": str(ano),  # <-- A CHAVE DO SUCESSO!
        "event": "MAR_9TGG963817D",
        "search[sex]": "M",
        "num_results": "5",  # Pede apenas os 5 primeiros
    }

    r = requests.get(url, headers=HEADERS, params=params)
    soup = BeautifulSoup(r.text, "html.parser")

    # Contagem de concluintes
    info_text = soup.find(class_=lambda c: c and "str_num" in str(c))
    total_str = info_text.get_text(strip=True) if info_text else "Sem contagem"

    # Primeiro atleta da lista
    primeiro_card = soup.select_one("li.list-group-item.row")
    nome = "-"
    tempo = "-"
    if primeiro_card and "list-group-header" not in primeiro_card.get(
        "class", []
    ):
        name_tag = primeiro_card.select_one("h4.list-field.type-fullname a")
        time_tag = primeiro_card.select_one("div.list-field.type-time")
        if time_tag and time_tag.select_one(".list-label"):
            time_tag.select_one(".list-label").decompose()
        nome = name_tag.get_text(strip=True) if name_tag else "-"
        tempo = time_tag.get_text(strip=True) if time_tag else "-"

    print(
        f"• Edição {ano} -> Total: {total_str:<16} | 1º Lugar: {nome:<28} ({tempo})"
    )


print("🔍 Testando edições históricas reais com event_main_group...\n")
for ano_teste in [1996, 1999, 2010, 2018, 2023, 2024]:
    testar_ano_com_group(ano_teste)

🔍 Testando edições históricas reais com event_main_group...

• Edição 1996 -> Total: 5626 Results     | 1º Lugar: -                            (-)
• Edição 1999 -> Total: 15319 Results    | 1º Lugar: -                            (-)
• Edição 2010 -> Total: 19973 Results    | 1º Lugar: -                            (-)
• Edição 2018 -> Total: 23963 Results    | 1º Lugar: -                            (-)
• Edição 2023 -> Total: 25803 Results    | 1º Lugar: -                            (-)
• Edição 2024 -> Total: 27999 Results    | 1º Lugar: -                            (-)


In [26]:
from bs4 import BeautifulSoup
import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:153.0) Gecko/20100101 Firefox/153.0"
}


def testar_sem_event_fixo(ano: int):
    url = "https://chicago-history.r.mikatiming.com/2024/"
    # NÃO passamos o 'event' fixo, deixamos o event_main_group comandar
    params = {
        "pid": "list",
        "event_main_group": str(ano),
        "search[sex]": "M",
        "num_results": "100",
        "page": "1",
    }

    r = requests.get(url, headers=HEADERS, params=params, timeout=10)
    soup = BeautifulSoup(r.text, "html.parser")

    cards = len(
        [
            c
            for c in soup.select("li.list-group-item.row")
            if "list-group-header" not in c.get("class", [])
        ]
    )
    info_text = soup.find(class_=lambda c: c and "str_num" in str(c))
    total_str = info_text.get_text(strip=True) if info_text else "Sem contagem"

    print(
        f"• Ano {ano} -> Total: {total_str:<18} | Atletas encontrados na Página 1: {cards}"
    )


print("🔍 Testando busca dinâmica sem 'event' fixo...\n")
for ano in [1996, 1997, 1998, 1999, 2005, 2010, 2018, 2024]:
    testar_sem_event_fixo(ano)

🔍 Testando busca dinâmica sem 'event' fixo...

• Ano 1996 -> Total: 5626 Results       | Atletas encontrados na Página 1: 100
• Ano 1997 -> Total: 9598 Results       | Atletas encontrados na Página 1: 100
• Ano 1998 -> Total: 10990 Results      | Atletas encontrados na Página 1: 100
• Ano 1999 -> Total: 15319 Results      | Atletas encontrados na Página 1: 100
• Ano 2005 -> Total: 18617 Results      | Atletas encontrados na Página 1: 100
• Ano 2010 -> Total: 19973 Results      | Atletas encontrados na Página 1: 100
• Ano 2018 -> Total: 23963 Results      | Atletas encontrados na Página 1: 100
• Ano 2024 -> Total: 28050 Results      | Atletas encontrados na Página 1: 100


In [27]:
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:153.0) Gecko/20100101 Firefox/153.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}


def testar_com_handshake_de_sessao(ano: int):
    session = requests.Session()
    base_url = "https://chicago-history.r.mikatiming.com/2024/"

    # 1. HANDSHAKE: Inicializa a sessão para o ano selecionado
    init_params = {
        "pid": "list",
        "event_main_group": str(ano),
    }
    r_init = session.get(
        base_url, headers=HEADERS, params=init_params, timeout=10
    )

    # 2. PAGINAÇÃO REAL: Página 1 com a sessão já calibrada
    list_params = {
        "pid": "list",
        "event_main_group": str(ano),
        "search[sex]": "M",
        "num_results": "100",
        "page": "1",
    }
    r_page1 = session.get(
        base_url, headers=HEADERS, params=list_params, timeout=10
    )
    soup = BeautifulSoup(r_page1.text, "html.parser")

    cards = len(
        [
            c
            for c in soup.select("li.list-group-item.row")
            if "list-group-header" not in c.get("class", [])
        ]
    )
    info_text = soup.find(class_=lambda c: c and "str_num" in str(c))
    total_str = info_text.get_text(strip=True) if info_text else "Sem contagem"

    print(
        f"🎯 Ano {ano} com Handshake -> Total: {total_str:<18} | Atletas Página 1: {cards} | Cookies: {list(session.cookies.keys())}"
    )


print("🔍 Testando inicialização de sessão por ano...\n")
for ano_teste in [1997, 1998, 1999, 2010, 2024]:
    testar_com_handshake_de_sessao(ano_teste)

🔍 Testando inicialização de sessão por ano...

🎯 Ano 1997 com Handshake -> Total: 9598 Results       | Atletas Página 1: 100 | Cookies: []
🎯 Ano 1998 com Handshake -> Total: 10990 Results      | Atletas Página 1: 100 | Cookies: []
🎯 Ano 1999 com Handshake -> Total: 15319 Results      | Atletas Página 1: 100 | Cookies: []
🎯 Ano 2010 com Handshake -> Total: 19973 Results      | Atletas Página 1: 100 | Cookies: []
🎯 Ano 2024 com Handshake -> Total: 28050 Results      | Atletas Página 1: 100 | Cookies: []


In [28]:
import math
import re
from bs4 import BeautifulSoup
import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:153.0) Gecko/20100101 Firefox/153.0"
}
BASE_URL = "https://chicago-history.r.mikatiming.com/2024/"


def checar_limite_paginas(ano: int, sexo: str = "M"):
    params = {
        "pid": "list",
        "event_main_group": str(ano),
        "search[sex]": sexo,
        "num_results": "100",
        "page": "1",
    }
    r = requests.get(BASE_URL, headers=HEADERS, params=params)
    soup = BeautifulSoup(r.text, "html.parser")

    # 1. Pega o total de resultados oficial
    info_text = soup.find(class_=lambda c: c and "str_num" in str(c))
    total_atletas = 0
    if info_text:
        m = re.search(r"(\d+)", info_text.text.replace(".", "").replace(",", ""))
        if m:
            total_atletas = int(m.group(1))

    total_paginas_esperadas = math.ceil(total_atletas / 100)
    print(
        f"📊 Chicago {ano} ({sexo}): {total_atletas:,} atletas -> Total REAL de páginas: {total_paginas_esperadas}"
    )

    # 2. Testa o que o servidor devolve além do limite (ex: página esperada + 5)
    page_alem = total_paginas_esperadas + 5
    params["page"] = str(page_alem)
    r_alem = requests.get(BASE_URL, headers=HEADERS, params=params)
    soup_alem = BeautifulSoup(r_alem.text, "html.parser")

    cards_alem = soup_alem.select("li.list-group-item.row")
    primeiro_nome = "-"
    if cards_alem:
        a_tag = cards_alem[0].select_one("h4.list-field.type-fullname a")
        if a_tag:
            primeiro_nome = a_tag.text.strip()

    print(
        f"⚠️ Teste na Página {page_alem} (além do fim): Devolveu {len(cards_alem)} cards. Primeiro atleta: '{primeiro_nome}'"
    )


checar_limite_paginas(1998, "M")

📊 Chicago 1998 (M): 10,990 atletas -> Total REAL de páginas: 110
⚠️ Teste na Página 115 (além do fim): Devolveu 2 cards. Primeiro atleta: '-'
